# 03 · Prompt variants

**Question:** does a prompt change improve triage without losing Critical recall?

`default` is whatever is in `api/src/saklolo/agents/prompts/` today.

History:
- `triage-p2-boundary`: rejected. Critical precision 93.9%, but Critical recall
  fell from 97.1% to 91.2%: the boundary overrode a head injury and a worsening update.
- `triage-p2-boundary-v2`: promoted. Lists what is always P1 before the boundary.
- `triage-p2-boundary-v3`: rejected. It narrowed v2's "water rising" rule to vulnerable
  people, so *"hanggang tuhod pa lang pero lumalalim"* left P1. But Critical recall fell
  from 100% to 97.1% (*"umaabot na sa 2nd floor"* became P3) and Critical precision from
  91.9% to 76.7%. Small wording changes move many unrelated messages on Nova Lite.

Variants live in `notebooks/prompts/`. Only a winner is copied to
`api/src/saklolo/agents/prompts/`.

In [ ]:
import lab

from saklolo.agents.prompts import PromptSet

In [ ]:
run_settings = lab.settings()
default = PromptSet.default()
variants = {"default": default} | {
    name: PromptSet(extract=default.extract, triage=lab.prompt_variant(name))
    for name in ["triage-p2-boundary-v3"]
}

In [ ]:
runs = {}
for label, prompts in variants.items():
    runs[label] = lab.run(lab.build_deps(run_settings, prompts=prompts), run_settings)
    print(label, "done,", len(runs[label].failed), "failed")

## Results
A variant that lowers Critical recall is rejected, even if precision improves.

In [ ]:
lab.compare(runs)

## Messages whose priority changed

In [ ]:
lab.priority_changes(runs["default"], runs["triage-p2-boundary-v3"])

## Decision
Write whether the variant wins and why. To promote it, copy it over
`api/src/saklolo/agents/prompts/triage.md` and run `make eval` (see `notebooks/README.md`).